In [1]:
# ==========================================================
# ACTIVIDAD 10. Consolidar los precios del universo
# Lee solo los archivos del universo, verifica su esquema,
# añade la columna ticker, recorta al periodo de estudio y
# une todo en un único DataFrame. No excluye nada todavía.
# ==========================================================
!pip install -q exchange_calendars

import time
from pathlib import Path

import exchange_calendars as xcals
import pandas as pd
from google.colab import drive
from huggingface_hub import snapshot_download

REPO_ID = "AmirTrader/YahooFinance"
COMMIT = "4c921233b7cc7b36c56891d21bed8c4f5010d7be"
INICIO = pd.Timestamp("2019-11-04")
FIN = pd.Timestamp("2026-08-31")
COLUMNAS = ["date", "open", "high", "low", "close", "adj_close", "volume"]

drive.mount("/content/drive")
CARPETA = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/00_control")
CARPETA_DATOS = Path("/content/drive/MyDrive/TFM_Regimenes_Riesgo/01_datos")
CARPETA_DATOS.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/datos_yahoo")                 # disco temporal de Colab

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)

# ----------------------------------------------------------
# 1. Tickers del universo (Actividad 9)
# ----------------------------------------------------------
universo = pd.read_csv(CARPETA / "universo_acciones.csv", dtype=str, keep_default_na=False)
tickers = sorted(universo["ticker"])
print("Tickers del universo:", len(tickers))

# ----------------------------------------------------------
# 2. Descargar SOLO esos archivos del commit congelado
# ----------------------------------------------------------
patrones = [f"data/daily/{t}.parquet" for t in tickers]
for intento in range(1, 6):
    try:
        snapshot_download(repo_id=REPO_ID, repo_type="dataset", revision=COMMIT,
                          allow_patterns=patrones, local_dir=LOCAL, max_workers=8)
        break
    except Exception as error:
        if "429" not in str(error) or intento == 5:
            raise
        print(f"Límite de descargas alcanzado (intento {intento}). Espero 5 minutos...")
        time.sleep(300)

ruta = lambda t: LOCAL / "data" / "daily" / f"{t}.parquet"
faltan = [t for t in tickers if not ruta(t).exists()]
print("Archivos que faltan tras la descarga:", faltan)

# ----------------------------------------------------------
# 3. Leer, verificar esquema, añadir ticker y recortar al periodo
# ----------------------------------------------------------
partes, esquema_distinto = [], []
for numero, ticker in enumerate(tickers, start=1):
    df = pd.read_parquet(ruta(ticker))
    if list(df.columns) != COLUMNAS:                 # verificación del esquema
        esquema_distinto.append(ticker)
        continue
    df["date"] = pd.to_datetime(df["date"], format="%Y-%m-%d")
    df = df[(df["date"] >= INICIO) & (df["date"] <= FIN)]
    df.insert(0, "ticker", ticker)                   # identificador desde el nombre del archivo
    partes.append(df)
    if numero % 500 == 0:
        print(f"Leídos {numero} de {len(tickers)}")

precios = pd.concat(partes, ignore_index=True)      # un único DataFrame
precios["ticker"] = precios["ticker"].astype("category")
precios = precios.sort_values(["ticker", "date"], ignore_index=True)

print("\n=== DATAFRAME CONSOLIDADO ===")
print("Archivos con esquema distinto:", esquema_distinto)
print("Filas:", len(precios), "| columnas:", list(precios.columns))
print("Tickers:", precios["ticker"].nunique(),
      "| fechas: de", precios["date"].min().date(), "a", precios["date"].max().date())
print(f"Memoria: {precios.memory_usage(deep=True).sum() / 1e6:.0f} MB")
print(precios.dtypes)
print(precios.head(3))

# ----------------------------------------------------------
# 4. Verificación fecha por fecha con el calendario oficial
# ----------------------------------------------------------
calendario = xcals.get_calendar("XNYS", start="2019-01-01", end="2026-12-31")
sesiones = calendario.sessions_in_range(INICIO, FIN)
precios["en_calendario"] = precios["date"].isin(sesiones)

calidad = precios.groupby("ticker", observed=True).agg(
    filas=("date", "size"),
    en_calendario=("en_calendario", "sum"),
)
calidad["faltan"] = len(sesiones) - calidad["en_calendario"]
calidad["sobran"] = calidad["filas"] - calidad["en_calendario"]
precios = precios.drop(columns="en_calendario")

print("\n=== FECHAS FRENTE AL CALENDARIO (", len(sesiones), "sesiones ) ===")
print("Tickers completos:", ((calidad["faltan"] == 0) & (calidad["sobran"] == 0)).sum())
print("Tickers con sesiones faltantes:", (calidad["faltan"] > 0).sum())
print("Tickers con fechas fuera del calendario:", (calidad["sobran"] > 0).sum())
print(calidad[(calidad["faltan"] > 0) | (calidad["sobran"] > 0)].head(20))

# ----------------------------------------------------------
# 5. Anomalías de precio y volumen (solo se cuentan)
# ----------------------------------------------------------
precios_cols = ["open", "high", "low", "close", "adj_close"]
rendimiento = precios.groupby("ticker", observed=True)["adj_close"].pct_change()
anomalias = pd.DataFrame({
    "ticker": precios["ticker"],
    "precio_no_positivo": (precios[precios_cols] <= 0).any(axis=1),
    "maximo_menor_minimo": precios["high"] < precios["low"],
    "cierre_fuera_rango": (precios["close"] > precios["high"] * 1.001)
                          | (precios["close"] < precios["low"] * 0.999),
    "volumen_cero": precios["volume"] == 0,
    "rendimiento_extremo": rendimiento.abs() > 0.5,
})
por_ticker = anomalias.groupby("ticker", observed=True).sum()
calidad = calidad.join(por_ticker)
calidad["max_rend_abs"] = rendimiento.abs().groupby(precios["ticker"], observed=True).max()

print("\n=== ANOMALÍAS (filas afectadas | tickers afectados) ===")
for columna in por_ticker.columns:
    print(f"{columna:22s} {int(por_ticker[columna].sum()):8d} | {(por_ticker[columna] > 0).sum():5d}")

print("\nTickers con más días de volumen cero:")
print(calidad.sort_values("volumen_cero", ascending=False)[["filas", "volumen_cero"]].head(10))
print("\nTickers con mayor rendimiento diario absoluto:")
print(calidad.sort_values("max_rend_abs", ascending=False)[["max_rend_abs", "rendimiento_extremo"]].head(10))

# ----------------------------------------------------------
# 6. Guardar el DataFrame consolidado y la tabla de calidad
# ----------------------------------------------------------
precios.to_parquet(CARPETA_DATOS / "precios_universo.parquet", index=False)
calidad.to_csv(CARPETA / "calidad_universo.csv")
print("\nGuardados: 01_datos/precios_universo.parquet -",
      round((CARPETA_DATOS / "precios_universo.parquet").stat().st_size / 1e6), "MB",
      "| 00_control/calidad_universo.csv")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ibis-framework 9.5.0 requires toolz<1,>=0.11, but you have toolz 1.2.0 which is incompatible.
Mounted at /content/drive
Tickers del universo: 2823


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2823 files:   0%|          | 0/2823 [00:00<?, ?it/s]

Archivos que faltan tras la descarga: []
Leídos 500 de 2823
Leídos 1000 de 2823
Leídos 1500 de 2823
Leídos 2000 de 2823
Leídos 2500 de 2823

=== DATAFRAME CONSOLIDADO ===
Archivos con esquema distinto: []
Filas: 4838622 | columnas: ['ticker', 'date', 'open', 'high', 'low', 'close', 'adj_close', 'volume']
Tickers: 2823 | fechas: de 2019-11-04 a 2026-08-31
Memoria: 281 MB
ticker             category
date         datetime64[ns]
open                float64
high                float64
low                 float64
close               float64
adj_close           float64
volume                int64
dtype: object

=== FECHAS FRENTE AL CALENDARIO ( 1714 sesiones ) ===
Tickers completos: 2823
Tickers con sesiones faltantes: 0
Tickers con fechas fuera del calendario: 0
Empty DataFrame
Columns: [filas, en_calendario, faltan, sobran]
Index: []

=== ANOMALÍAS (filas afectadas | tickers afectados) ===
precio_no_positivo         1653 |     3
maximo_menor_minimo           0 |     0
cierre_fuera_rango    